# 09: Multi-Core Synchronization

## Objective
Use multiple tProc cores simultaneously, manage triggers, and handle cross-core dependencies.

In [ ]:
BITSTREAM_PATH = "/path/to/firmware.bit"
USE_PROXY = False
PROXY_IP = "192.168.1.100"

from qick import QickSoc, QickConfig
import numpy as np
import matplotlib.pyplot as plt

if USE_PROXY:
    from qick.pyro import make_proxy
    soc = make_proxy(PROXY_IP)
else:
    soc = QickSoc(bitfile=BITSTREAM_PATH)

print(f"Available tProc cores: {soc.num_tprocs}")

In [ ]:
config0 = QickConfig(core=0)
config1 = QickConfig(core=1)

config0.pulse(ch=0, style="const", length=100, gain=0.5)
config0.pulse(ch=0, style="const", length=100, gain=0)

config1.pulse(ch=1, style="const", length=100, gain=0.3)
config1.pulse(ch=1, style="const", length=100, gain=0)

soc.load_config(config0, core=0)
soc.load_config(config1, core=1)

soc.start_core(0)
soc.start_core(1)

print("Both cores running")

In [ ]:
# Core 0 generates trigger for Core 1
config0 = QickConfig(core=0)
config0.pulse(ch=0, style="const", length=50, gain=0.5)
config0.trigger(t=100, length=10, dest=1)

config1 = QickConfig(core=1)
config1.wait_trigger(t=0, timeout=1000)
config1.pulse(ch=1, style="const", length=50, gain=0.5)

soc.load_config(config0, core=0)
soc.load_config(config1, core=1)
soc.start_core(0)
soc.start_core(1)

print("Trigger-based synchronization active")

In [ ]:
# Core 0 waits for Core 1 to finish
config0 = QickConfig(core=0)
config0.pulse(ch=0, style="const", length=50, gain=0.5)
config0.wait_core(core=1, t=100)  # Wait for core 1 at t=100
config0.pulse(ch=0, style="const", length=50, gain=0.3)

config1 = QickConfig(core=1)
config1.pulse(ch=1, style="const", length=200, gain=0.5)

soc.load_config(config0, core=0)
soc.load_config(config1, core=1)
soc.start_core(0)
soc.start_core(1)

print("Core 0 waits for Core 1 - check timing")